# **Step2_AI 강사 Agent v2.0**

## **0. 미션**

### **미션③ : 모듈 고도화1**
다음 항목에 대해서 조 상황에 맞게 선택적으로 고도화 합니다.

* 입력 프롬프트 추가 : 강의 목소리, 톤 조절, 강의 스타일 지침
* 정보 분해 : 여러 슬라이드를 저장하고 관리하기 위한 폴더 준비, 불필요한 정보 제거, 제목 추출
* 페이지별 내용 생성 : 슬라이드 내 정보 뿐만 아니라 부연 설명을 위한 검색 기능 추가
* 강의 스크립트 생성 : 전체 강의 내용을 참조하여 강의 흐름을 구상하고,
                                   현재 페이지 강의 스크립트 작성
* 내용 검토 : 페이지 내용과 강의 스크립트 비교, 강의 스크립트 흐름 적절한지 검토




### **미션④ : AI 강사 Agent 완성**
* 모듈 고도화2(다음 항목에 대해서 조 상황에 맞게 선택적으로 고도화 합니다.)
    * 음성 변환 : 강의 목소리, 톤 조절 음성 변환
    * 영상 제작 :
        * 각 페이지 : 음성과 슬라이드 스냅샷 이미지 합성하여 영상 제작하기
        * 전체 강의 영상 : 각 슬라이드 강의를 전체 슬라이드 강의 영상으로 합치기
* 웹 화면 연결(gradio)
    * 음성 변환 : 강의 목소리, 톤 조절 프롬프트 기반 음성 변환
* 전체 Agent 그래프 구축
    * 전체를 Agent 그래프로 엮기


## Agent workflow image

> Embedded base64 image was removed for a cleaner public repository. See `docs/agent-workflow.md` for the workflow summary.


## **1. 환경준비**

### (1) 구글 드라이브

* 구글 드라이브 폴더 생성
    * 새 폴더 `proj1_agent`를 생성하고
    * 제공 받은 파일을 업로드

* 구글 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### (2) 한글 폰트 준비
* 한글 폰트 설치 및 설정 코드
* 영상 제작 등 한글 사용시 필요

In [ ]:
%%bash
set -e

# (1) 필수 패키지
sudo apt-get update -y
sudo apt-get install -y \
  libreoffice poppler-utils poppler-data ffmpeg locales fontconfig xvfb \
  fonts-noto-core fonts-noto-cjk fonts-noto-cjk-extra

# (2) 로케일
sudo sed -i 's/^# *ko_KR.UTF-8 UTF-8/ko_KR.UTF-8 UTF-8/' /etc/locale.gen
sudo locale-gen ko_KR.UTF-8
sudo update-locale LANG=ko_KR.UTF-8

# (3) 기존에 만들었던 사용자 fonts.conf(전체 덮어쓰기) 제거
rm -f ~/.config/fontconfig/fonts.conf || true

# (4) 시스템 fontconfig에 로컬 룰로만 추가 (기본 설정은 그대로 사용)
sudo tee /etc/fonts/local.conf >/dev/null <<'EOF'
<?xml version="1.0"?>
<!DOCTYPE fontconfig SYSTEM "fonts.dtd">
<fontconfig>

  <!-- 한국어는 Noto Sans KR 우선 -->
  <match target="pattern">
    <test name="lang" compare="contains"><string>ko</string></test>
    <edit name="family" mode="prepend" binding="strong">
      <string>Noto Sans KR</string>
      <string>Noto Sans CJK KR</string>
      <string>Noto Sans</string>
    </edit>
  </match>

  <!-- 흔한 한글 폰트명을 Noto Sans KR로 매핑 -->
  <alias><family>Malgun Gothic</family><prefer><family>Noto Sans KR</family></prefer></alias>
  <alias><family>맑은 고딕</family><prefer><family>Noto Sans KR</family></prefer></alias>
  <alias><family>Apple SD Gothic Neo</family><prefer><family>Noto Sans KR</family></prefer></alias>
  <alias><family>AppleGothic</family><prefer><family>Noto Sans KR</family></prefer></alias>

</fontconfig>
EOF

# (5) 캐시 완전 재생성
rm -rf ~/.cache/fontconfig
sudo fc-cache -r
fc-cache -f -v >/dev/null

# (6) 검증
echo "---- fc-match ko ----"
fc-match "sans-serif:lang=ko"
echo "---- noto candidates ----"
fc-list | grep -i -E "noto.*(sans|cjk).*kr" | head -n 30 || true

### (3) 라이브러리

* 필요한 라이브러리 설치

In [ ]:
!pip install langchain-openai langchain-community python-pptx pillow gradio langchain-tavily tavily-python -q

* 라이브러리 로딩

In [ ]:
import os, re, textwrap, subprocess, json, base64, mimetypes, shlex
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Optional, TypedDict, Any
from PIL import Image, ImageDraw
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE
from openai import OpenAI
from google.colab import files
from IPython.display import Audio, display, Video

from langchain_tavily import TavilySearch
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

### (4) API Key 확인
* 구글드라이브에 생성한 폴더 `proj1_agent` 에서
* api_key.txt 파일 안에 각자 발급 받은 키를 저장합니다.
    * **OPENAI_API_KEY**
    * **TAVILY_API_KEY**

In [ ]:
def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

path = '/content/drive/MyDrive/proj1_agent/'
# API 키 로드 및 환경변수 설정
load_api_keys(path + 'api_key.txt')

⚠️ 아래 코드셀은, 실행해서 key가 제대로 보이는지 확인하고 삭제하세요.

In [ ]:
# 공개 저장소에서는 API Key 값을 출력하지 않습니다.
required_keys = ['OPENAI_API_KEY', 'TAVILY_API_KEY']
missing = [key for key in required_keys if not os.environ.get(key)]
print('API keys loaded' if not missing else f'Missing keys: {missing}')


### (5) 유용한 함수들 제공
* 다음은 프로젝트를 수행하는데 유용한 함수들입니다.
* 이 함수들의 내용을 확인하고 필요시 활용하여 개인 과제를 수행합니다.(꼭 활용해야 하는 것은 아닙니다.)

In [ ]:
# node로 쓰는 것은 아니다. 함수로 써라!

* 공백 제거 함수

In [ ]:
def clean_text(s):
    return re.sub(r"\s+", " ", s).strip()

* 긴 문자열을 문장 단위로 나누는 문장 분리기

In [ ]:
def split_sents(t: str) -> List[str]:
    parts = re.split(r'([\.?!])', t)
    merged = []
    for i in range(0, len(parts)-1, 2):
        sent = (parts[i] + parts[i+1]).strip()
        if sent: merged.append(sent)
    if len(parts) % 2 == 1 and parts[-1].strip():
        merged.append(parts[-1].strip())
    return [s for s in merged if s]

* 오디오 길이 계산

In [ ]:
def ffprobe_duration(path: str) -> float:
    out = subprocess.check_output([
        "ffprobe","-v","error","-show_entries","format=duration",
        "-of","default=noprint_wrappers=1:nokey=1", path]).decode().strip()
    return float(out)

* 이미지를 base64로 변환

In [ ]:
def img_to_data_url(path: str) -> str:
    mime = mimetypes.guess_type(path)[0] or "image/png"
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime};base64,{b64}"

* 배경 이미지와 오디오 합쳐서 MP4 영상 만들기

In [ ]:
def render_mp4(image_path: str, audio_path: str, out_mp4: str,
               width=1920, height=1080,
               highlight_word=None, highlight_time=2.0):

    dur = ffprobe_duration(audio_path)

    # 1. 기본 필터 (스케일링 및 검은색 배경 패딩)
    vf = (f"scale={width}:{height}:force_original_aspect_ratio=decrease,"
          f"pad={width}:{height}:(ow-iw)/2:(oh-ih)/2:color=black")

    # 2. 시각 효과 필터 추가 (조건부)
    if highlight_word:
        # 강조 시작 시간과 종료 시간(2초간 노출) 설정
        start_t = highlight_time
        end_t = start_t + 2.0

        # 빨간색 강조 원(●)과 텍스트를 화면 중앙에 띄우는 필터
        # alpha='if(lt(t,st),0,if(lt(t,et),1,0))' 형식으로 페이드 효과도 가능하지만 단순하게 구성
        # overlay_vf = (
        #     f",drawtext=text='●':fontcolor=red@0.6:fontsize=160:"
        #     f"x=(w-tw)/2:y=(h-th)/2-100:enable='between(t,{start_t},{end_t})',"
        #     f"drawtext=text='{highlight_word}':fontcolor=white:fontsize=80:box=1:boxcolor=black@0.4:boxborderw=10:"
        #     f"x=(w-tw)/2:y=(h-th)/2+50:enable='between(t,{start_t},{end_t})'"
        # )

        #주변 하이라이트
        overlay_vf = (
        f",drawtext=text='{highlight_word}':fontcolor=yellow:fontsize=85:fontfile={font_path if 'font_path' in locals() else 'Sans'}:"
        f"box=1:boxcolor=black@0.5:boxborderw=30:borderw=3:bordercolor=yellow@0.3:" # 배경 박스 및 은은한 외곽선
        f"x=(w-tw)/2:y=(h-th)/2:enable='between(t,{start_t},{end_t})':"
        f"alpha='if(lt(t,{start_t}+0.3),(t-{start_t})/0.3,if(lt(t,{end_t}-0.3),1,({end_t}-t)/0.3))'" # Fade In/Out 애니메이션
        )
        vf += overlay_vf

    # FFmpeg 명령
    cmd = ["ffmpeg", "-y",
            "-loop", "1", "-i", image_path,   # 정지 이미지 입력
            "-i", audio_path,                 # 오디오 입력
            "-t", str(dur),                   # 길이 = 오디오 길이
            "-vf", vf,                        # 비디오 필터
            "-c:v", "libx264", "-preset", "veryfast", "-crf", "20",
            "-c:a", "aac", "-b:a", "192k",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",        # 웹/브라우저 재생 친화
            out_mp4]
    subprocess.check_call(cmd)  # 외부 프로그램(FFmpeg)을 파이썬 프로세스에서 실행하고, 성공했는지 확인

* ppt 슬라이드를 배경 이미지로 저장
    * 강의 영상에서 사용할 배경 이미지 생성하기
    * ppt를 pdf로 변환한 뒤 다시 이미지로 변환

In [ ]:
import os, subprocess, uuid
from pathlib import Path

def export_slide_as_png(state: dict, dpi: int = 220) -> dict:
    work_dir = Path(state["work_dir"]).expanduser().resolve()
    work_dir.mkdir(parents=True, exist_ok=True)

    pptx = Path(state["pptx_path"]).expanduser().resolve()
    if not pptx.exists():
        raise FileNotFoundError(f"PPTX 없음: {pptx}")

    idx = int(state.get("slide_index", 0))  # 0-based
    page_no = idx + 1
    out_prefix = work_dir / "slide_img"

    # LibreOffice 프로필(캐시) 꼬임 방지
    lo_profile = f"file:///tmp/lo_profile_{uuid.uuid4().hex}"

    # fontconfig/locale 강제
    env = os.environ.copy()
    env.update({
        "LANG": "ko_KR.UTF-8",
        "LC_ALL": "ko_KR.UTF-8",
        "FONTCONFIG_PATH": str(Path("~/.config/fontconfig").expanduser()),
        "FONTCONFIG_FILE": str(Path("~/.config/fontconfig/fonts.conf").expanduser()),
        "HOME": str(Path("~").expanduser()),
        "XDG_CACHE_HOME": str(Path("~/.cache").expanduser()),
        "SAL_USE_VCLPLUGIN": "gen",
    })

    def run_lo_convert(convert_to: str):
        # X11 요구 해결: xvfb-run로 감싼다 (Colab 필수급)
        cmd = [
            "xvfb-run", "-a",
            "soffice", "--headless", "--nologo", "--nofirststartwizard", "--norestore",
            f"-env:UserInstallation={lo_profile}",
            "--convert-to", convert_to,
            "--outdir", str(work_dir),
            str(pptx),
        ]
        return subprocess.run(cmd, capture_output=True, text=True, env=env)

    # --- A) PPTX → PNG ---
    before_png = set(work_dir.glob("*.png"))
    res_png = run_lo_convert("png:impress_png_Export")

    created_png = [p for p in work_dir.glob("*.png") if p not in before_png]
    candidate = None

    exact = [p for p in created_png if p.stem.endswith(f"-{page_no}")]
    if exact:
        candidate = max(exact, key=lambda p: p.stat().st_mtime)
    elif created_png:
        candidate = max(created_png, key=lambda p: p.stat().st_mtime)

    if candidate and candidate.exists():
        state["slide_image"] = str(candidate)
        return state

    # --- B) 폴백: PPTX → PDF → PNG ---
    before_pdf = set(work_dir.glob("*.pdf"))
    res_pdf = run_lo_convert("pdf:impress_pdf_Export")

    target_pdf = work_dir / f"{pptx.stem}.pdf"
    created_pdf = [p for p in work_dir.glob("*.pdf") if p not in before_pdf]

    if target_pdf.exists():
        pdf_path = target_pdf
    elif created_pdf:
        pdf_path = max(created_pdf, key=lambda p: p.stat().st_mtime)
    else:
        print("LibreOffice PPTX→PDF 변환 실패")
        print("stdout:", res_pdf.stdout)
        print("stderr:", res_pdf.stderr)
        raise RuntimeError("PPTX → PDF 변환 실패")

    ppm_cmd = [
        "pdftoppm",
        "-f", str(page_no), "-l", str(page_no),
        "-png", "-r", str(dpi),
        str(pdf_path),
        str(out_prefix)
    ]
    res2 = subprocess.run(ppm_cmd, capture_output=True, text=True, env=env)

    png_path = Path(f"{out_prefix}-{page_no}.png")
    if not png_path.exists():
        print("pdftoppm 변환 실패")
        print("stdout:", res2.stdout)
        print("stderr:", res2.stderr)
        raise RuntimeError("PDF → PNG 변환 실패")

    state["slide_image"] = str(png_path)
    return state

* 영상 합치기 : 여러 영상 경로를 리스트로 입력 받아 합치기

In [ ]:
def concat_videos_ffmpeg(video_paths: List[str], out_path: str, reencode: bool=False):
    list_path = out_path + ".txt"
    with open(list_path, "w", encoding="utf-8") as f:
        for v in video_paths:
            f.write(f"file '{os.path.abspath(v)}'\n")
    if reencode:
        cmd = [
            "ffmpeg","-y","-safe","0","-f","concat","-i",list_path,
            "-vf","format=yuv420p",
            "-c:v","libx264","-preset","veryfast",
            "-c:a","aac","-b:a","192k",
            out_path
        ]
    else:
        cmd = ["ffmpeg","-y","-safe","0","-f","concat","-i",list_path,"-c","copy",out_path]
    subprocess.check_call(cmd)

-----

**[주의!]**🚨🚨🚨🚨🚨
* 아래 제공되는 코드 혹은 노드를 그대로 사용할 경우, 오류가 발생될 수 있습니다.
* 코드 내용을 반드시 확인하고, 여러분의 프로젝트에 맞게 수정해서 사용하세요.

-----

## **2. 미션③ : 모듈 고도화1**
(다음 항목에 대해서 조 상황에 맞게 선택적으로 고도화 합니다.)
* 입력 프롬프트 추가 : 강의 목소리, 톤 조절, 강의 스타일 지침
* 정보 분해 : 여러 슬라이드를 저장하고 관리하기 위한 폴더 준비, 불필요한 정보 제거, 제목 추출
* 페이지별 내용 생성 : 슬라이드 내 정보 뿐만 아니라 부연 설명을 위한 검색 기능 추가
* 강의 스크립트 생성 : 전체 강의 내용을 참조하여 강의 흐름을 구상하고,
                                   현재 페이지 강의 스크립트 작성
* 내용 검토 : 페이지 내용과 강의 스크립트 비교, 강의 스크립트 흐름 적절한지 검토

### (1) 파일 입력

* ppt 파일



In [ ]:


# 파일 업로드
uploaded = files.upload()
pptx_path = list(uploaded.keys())[0]

In [ ]:
# 사용자 프롬프트
USER_PROMPT = {
    "voice": "alloy",
    "tone": "친절하고 명료한 강의 톤",
    "style": "예시와 핵심 요점 중심"
}

# 출력 dir 만들기
WORK_DIR = "./step2_output"
MEDIA_DIR = "./step2_output/media"
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(MEDIA_DIR, exist_ok=True)

### (2) State 선언

* 각 노드의 입출력 관리를 위한 State 구성
    * 각 함수(노드)에서 채워가며 관리해야 할 정보를 도출
    * 이를 하나의 State로 정의


In [ ]:
# State 정의 및 초기화
class State(TypedDict, total=False): # total=False는 TypedDict에서 모든 키를 선택(optional)으로 취급
    # 입력/기본
    pptx_path: str       # 처리할 PPTX 파일의 경로
    work_dir: str        # 생성된 결과물(audio, video)파일을 저장하는 경로
    prompt: Dict         # 사용자 맞춤형 지침(말투, 스타일, 목소리 종류 등)을 담은 딕셔너리
                         # 페이지 요약 시 '강의 스타일/톤'을 결정하고, 대본 작성 시 '말투'를 결정하며, TTS 생성 시 'voice'를 선택하는 데 사용
    slide_index: int     # 현재 처리 중인 슬라이드 인덱스 번호(0-based). 개인 과제는 0 고정.

    # 추출/생성/미디어 변수 추가
    # 추출 산출물: 슬라이드 원본에서 뽑아낸 데이터
    texts: List[str]
    tables: List[List[List[str]]]
    images: List[str]
    slide_image: List[str]
    search_result : List[str] # 외부검색 결과 저장
    feedback: str                   # 사용자 피드백

    # 생성 산출물: LLM이 이전 단계의 정보를 바탕으로 새롭게 만들어낸 텍스트
    page_content: List[str] # 슬라이드의 텍스트, 표, 이미지 정보를 바탕으로 작성된 강의 요약 문단 리스트
    script: List[str] # 요약된 page_content를 바탕으로 작성된 실제 발표용 대본 리스트

    # 미디어 산출물
    audio: str # 생성된 음성 파일(.mp3)의 저장 경로
    video_path: List[str] # List로 변경

    #0427추가: 파싱 결과 (다중 슬라이드)
    slides: List[Dict[str, Any]] # [{index,title,texts,tables,images,snap}]
    n_slides: int # 총 슬라이드 장수

    # 임시(반복 내부 사용)
    cur_search_context: str #현재 처리중인 슬라이드 key
    cur_page_content: str #현재 처리중인 슬라이드 key: 슬라이드의 텍스트, 표, 이미지 정보를 바탕으로 작성된 강의 요약 문단
    cur_script: str #현재 처리중인 슬라이드 key: 요약된 page_content를 바탕으로 작성된 실제 발표용 대본
    cur_audio: str # 현재 슬라이드에서 생성된 음성 파일(.mp3)의 저장 경로
    cur_video: str # 현재 슬라이드에서 이미지와 소리가 합쳐진 최종 동영상 파일(.mp4)의 경로 (v1.0에서의 video_path에 해당)

    final_video: str # 최종 video 합친 path

    #HighLight
    cur_highlight_word: str
    cur_highlight_time: float

    # Vector DB 및 퀴즈 관련
    vectorstore: Any       # 생성된 Chroma DB 객체
    quiz_results: List[str] # 생성된 객관식 퀴즈 텍스트
    num_quizzes_to_generate: int # 생성할 퀴즈 개수

    # 자막
    subtitles: List[Dict] #[{"text": "...", ...}]
    cur_subtitles: List[Dict] # 현재 슬라이드 자막 리스트


In [ ]:
# 초기 state 설정

state: State = {"pptx_path": pptx_path,
                "work_dir": WORK_DIR,
                "prompt": USER_PROMPT}
state

In [ ]:
# 모델 준비
LLM_MODEL = "gpt-4o-mini"
TTS_MODEL = "gpt-4o-mini-tts"

In [ ]:
llm = ChatOpenAI(model=LLM_MODEL, temperature=0.3)

### (3) ppt 정보 분해
* 목적: 전체 슬라이드의 텍스트/표/이미지/스냅샷 수집 → state["slides"] 적재
* 입력: pptx_path, work_dir
* 출력: 슬라이드마다, text, image, text, 스냅(스크린샷) 등 저장
* 처리:
    * python-pptx로 텍스트/표/이미지 추출
    * 제목 추출
    * export_slide_as_png로 snap 생성

* 노드 함수 생성 (일부 제공)

In [ ]:
def node_parse_all(state: State) -> State:
    '파서 노드 내부에서 반복문으로 모든 슬라이드 텍스트/표/이미지 + 스냅샷 생성'
    pres = Presentation(state["pptx_path"])
    work_dir = state["work_dir"]
    media_dir = os.path.join(work_dir, "media")
    os.makedirs(media_dir, exist_ok=True)
    slides_out: List[Dict[str, Any]] = []  # 함수 내에서 결과물 저장용

    # 반복문으로 각 슬라이드 마다 텍스트/표/이미지 추출
    for idx, slide in enumerate(pres.slides, start=1):
        texts = []
        tables = []
        images = []
        title = ""
        # 텍스트/표/이미지 추출
        # 텍스트 추출
        for shape in slide.shapes:
          if shape.shape_type == MSO_SHAPE_TYPE.PLACEHOLDER: # 추가된 부분
            if shape.placeholder_format.idx == 0:
              title = clean_text(shape.text_frame.text)
          if shape.has_text_frame :
            print(f'[{idx}] text')
            txt = "\n".join(p.text for p in shape.text_frame.paragraphs)
            print(txt)
            splited = split_sents(txt) # 문장 단위 추출
            # cleaned = clean_text(splited) # clean_text로 정제
            cleaned = [clean_text(s) for s in splited] # 각 문장별로 정제
            texts.append(cleaned)

          # 표 추출
          if shape.shape_type == MSO_SHAPE_TYPE.TABLE :
            print(f'[{idx}] table')
            tbl = [[clean_text(c.text) for c in r.cells] for r in shape.table.rows]
            print(tbl)
            tables.append(tbl)

          # 이미지 추출
          if shape.shape_type == MSO_SHAPE_TYPE.PICTURE :
            print(f'[{idx}] image')
            ext = shape.image.ext              # 이미지의 원본 확장자를 얻는다(예: png, jpg).
            path = os.path.join(media_dir, f"slide{idx}_img_{len(images)}.{ext}")  # 파일 저장 경로
            print(path)
            images.append(path)
            with open(path, "wb") as f:
              f.write(shape.image.blob)         # 파일 저장

        # 스냅샷(export_slide_as_png 사용)
        tmp_state = {"pptx_path": state["pptx_path"], "work_dir": work_dir, "slide_index": idx-1}
        snap_state = export_slide_as_png(tmp_state)
        snap = snap_state["slide_image"]

        # 결과물 저장
        slides_out.append({
            "index": idx,
            "title": title,
            "texts": texts,
            "tables": tables,
            "images": images,
            "snap": snap,
        })

    state["slides"] = slides_out
    state["n_slides"] = len(slides_out)
    state["slide_index"] = 0
    state["video_path"] = [None] * len(slides_out)
    return state

* 노드 테스트

In [ ]:
state : State = node_parse_all(state)

state

In [ ]:
# 슬라이드 이미지 조회 테스트

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

n = state["n_slides"]
cols = 2
rows = (n + cols - 1) // cols

plt.figure(figsize=(15, 5 * rows))

for idx, slide in enumerate(state["slides"]):
    img_path = slide.get("snap")
    title = slide.get("title", f"Slide {idx+1}")

    plt.subplot(rows, cols, idx + 1)
    if img_path and os.path.exists(img_path):
        img = mpimg.imread(img_path)
        plt.imshow(img)
    else:
        plt.text(0.5, 0.5, f"이미지 없음\n{img_path}", ha='center', va='center')

    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# my test

# node_parse_all 함수를 실행하여 슬라이드를 파싱하고 state를 업데이트
state = node_parse_all(state)

# 슬라이드 이미지 조회 테스트
# 한번 실행해서 제대로 조회가 되는지 테스트해보기.
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# (선택) 한글 깨짐 방지를 위한 폰트 설정 (Colab 환경 등에서 필요 시 주석 해제)
# plt.rc('font', family='NanumGothic') # Windows: 'Malgun Gothic', Mac: 'AppleGothic'
# plt.rcParams['axes.unicode_minus'] = False

n = state["n_slides"]
cols = 2
rows = (n + cols - 1) // cols

plt.figure(figsize=(15, 5 * rows))

for idx, slide in enumerate(state["slides"]):
    img_path = slide.get("snap")
    title = slide.get("title", f"Slide {idx+1}")

    plt.subplot(rows, cols, idx + 1)
    if img_path and os.path.exists(img_path):
        img = mpimg.imread(img_path)
        plt.imshow(img)
    else:
        plt.text(0.5, 0.5, f"이미지 없음\n{img_path}", ha='center', va='center')

    plt.axis('off')

plt.tight_layout()
plt.show()

### (4) 내용 생성

* 목적: 슬라이드 내용을 text로 정리
* 입력 : text, image, 표, 슬라이드 제목
* 출력 : 슬라이드 설명문
* 처리
    * 슬라이드 제목으로 SerpAPI 검색 및 요약
    * text, image, 표에 대한 설명문 생성
    * 전체 설명문 작성

#### **1) 외부 검색 노드**


### 페이지별 내용 생성 : 슬라이드 내 정보 뿐만 아니라 부연 설명을 위한 검색 기능 추가

In [ ]:
from langgraph.prebuilt import ToolNode
from langchain_community.tools.tavily_search import TavilySearchResults

# 1. 도구(Tools) 리스트 준비 (제시해주신 코드 반영)
tavily_tool = TavilySearchResults(max_results=3)
tools = [tavily_tool] # 필요하다면 [tavily_tool, wiki_tool, arxiv_tool] 처럼 확장 가능
tool_node = ToolNode(tools)
llm_with_tool = llm.bind_tools(tools)

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import SystemMessage, HumanMessage

def node_tool_search(state : State) -> State:
    """현재 슬라이드 제목/키워드로 외부 검색하여 추가 정보 수집"""
    # 현재 처리 중인 슬라이드 정보 가져오기
    idx = state["slide_index"]
    current_slide = state["slides"][idx]
    title = current_slide.get("title", "")

    # 제목이 없으면 검색 스킵
    if not title:
      state["search_result"] = []
      return state

    # LLM에게 검색 쿼리 생성 요청 (제목 기반으로 적절한 검색어 도출)
    messages = f"""
    다음 슬라이드 제목을 보고 부연 설명에 필요한 핵심 검색어를 한 문장으로 만들어줘.
    제목: {title}
    검색어만 출력해. 다른 말은 하지 마.
    """
    query_msg = llm.invoke(messages)
    search_query = query_msg.content.strip()
    print(f"[tool_search] slide {idx+1} | 제목: {title} | 검색어: {search_query}")

    # ToolNode 대신 tavily_tool 직접 호출
    raw_results = tavily_tool.invoke({"query": search_query})

    # 결과 파싱 - TavilySearch는 list of dict를 반환
    results_text = []
    if isinstance(raw_results, list):
        for item in raw_results:
            content = item.get("content", "")
            url = item.get("url", "")
            if content:
                results_text.append(f"[출처: {url}]\n{content}")
    elif isinstance(raw_results, str):
        results_text.append(raw_results)

    state["search_result"] = results_text
    print(f"[tool_search] 검색 결과 {len(results_text)}건 저장 완료")
    return state

In [ ]:
# 테스트
state : State = node_tool_search(state)
state

#### **2) 내용 생성 함수**

In [ ]:
def node_generate_page(state: State) -> State:
    # 1. 슬라이드에서 사용할 정보 준비 ---------------
    idx = state["slide_index"]
    current_slide = state["slides"][idx]
    title = current_slide.get("title", f"Slide {idx + 1}")
    texts = current_slide.get("texts", [])
    tables = current_slide.get("tables", [])

    # ## 슬라이드 표: 첫 표 6행까지만 문자열화

    table_snip = ""
    if tables:
        try:
            table_snip = "\n".join([" | ".join(map(str, r)) for r in tables[0][:6]])
        except Exception:
            table_snip = str(tables[0][:6])
    ## 슬라이드 이미지
    images = list(current_slide.get("images", []) or [])
    slide_img = current_slide.get("snap") # 슬라이드 스냅샷이 있으면 가장 앞에 우선 포함
    if slide_img:
        if isinstance(slide_img, list):
            if slide_img and slide_img[0] not in images:
                images.insert(0, slide_img[0])
        else:
            if slide_img not in images:
                images.insert(0, slide_img)

    ## 사용자가 입력한 강의 스타일
    prompt = state["prompt"]

    # 2. 메시지 구성 -----------------------
    ## 시스템 메시지
    system_msg = SystemMessage(content=f"""
    규칙 :
    - {prompt['style']} 으로 슬라이드의 텍스트, 표, 이미지를 분석하여 요약해주세요.
    - 과장 금지
    - 4~6문장
    - 불릿 금지
    - 모르는 내용 창작 금지
    - 이미지가 있으면 이미지 내용도 설명에 포함
    """)

    ## HumanMessage : 리스트로 구성해야 멀티모달 입력 가능
    user_text = []
    user_text.append(f"[텍스트]\n{texts if texts else ' (없음) '}")
    user_text.append(f"[표요약]\n{table_snip if table_snip else'(없음)'}")
    user_text = "\n\n".join(user_text)

    ### 멀티모달 HumanMessage 구성
    human_msg = [{"type" : "text", "text":user_text}]

    MAX_IMGS = 3
    for image in images[:MAX_IMGS]:
      try:
        data_url = img_to_data_url(image) # base 64 변환
        human_msg.append({"type": "image_url", "image_url": {"url": data_url}})
      except Exception:
        pass

    # 3. llm.invoke()  ----------------
    response = llm.invoke([system_msg, HumanMessage(human_msg)])
    state["page_content"] = response.content

    return state

* 노드 테스트

In [ ]:
# 테스트
state : State = node_generate_page(state)
print(state["page_content"][:200], "...")

### (5) 강의 스크립트

* 목적 : 슬라이드 요약(page_content)을 기반으로 60~90초 분량의 발표 대본(스크립트)을 작성하고 저장
* 입력:
    * 슬라이드 설명문
    * tone, 말투 지시 프롬프트
    * 스크립트 저장 디렉토리
    * 이전 페이지의 강의 스크립트(맥락/흐름 파악용)
* 출력: 생성된 발표 스크립트

* 노드 함수 생성

In [ ]:
def node_generate_script_ctx(state: State) -> State:
    # 1. 슬라이드에서 사용할 정보 준비 ---------------
    content1 = state["page_content"]
    # content2 = state["search_result"]
    search_data = state.get("search_result", [])
    content2 = "\n".join(search_data) if isinstance(search_data, list) else str(search_data)
    prompt = state["prompt"]

    # 2. 메시지 구성 -----------------------
    sys_msg = """
# 역할 : 너는 발표 대본 작성 보조 에이전트다.

# 지침
- 제공된 내용을 바탕으로 발표자가 실제로 말할 수 있는 자연스럽고 구어체에 가까운
한국어 발표 스크립트를 작성한다.
- 또한 강의 대본을 작성할 때, 화면에서 가장 강조해야 할 핵심 키워드 하나와 그 키워드가 언급될 예상 시간(초)을 반드시 포함한다.
반드시 아래의 JSON 형식으로만 응답하세요. 다른 설명은 금지합니다.
{{
"script": "실제 강사가 말할 대본 내용",
"highlight_word": "강조할 핵심 키워드(반드시 영어로 작성하세요. 예: Monitoring)",
"highlight_time": 3.5
}}
- 결과는 60~90초 분량으로 작성한다.
- 구성은 다음 흐름을 따른다:
  1) 인트로 1문장
  2) 핵심 내용을 청자가 이해하기 쉽도록 풀어서 설명
  3) 마무리 1문장
- 핵심 개념은 짧게 끊지 말고, 맥락과 이유를 포함해 한 번 더 풀어 설명한다.
- 예시나 상황을 간단히 덧붙여 설명의 이해도를 높인다.
- 숫자와 용어는 원문 기준으로 정확하게 유지한다.
- 불필요한 수식어와 반복은 줄이되, 설명이 부족하지 않도록 자연스럽게 보완한다.
- 불릿, 번호 목록, 코드블록은 사용하지 않는다.
- 대본(script)은 한국어 구어체로, 강조 키워드(highlight_word)는 반드시 영어로 작성한다.
"""

    human_msg = f"""
다음 내용을 토대로 발표 스크립트를 한국어로 작성해줘.

- 말투: {prompt.get('tone')}

[내용]
{content1}
{content2}
"""

    # 3. llm.invoke() ------------------
    response = llm.invoke([SystemMessage(content=sys_msg),
                           HumanMessage(content=human_msg)])

    raw_res = response.content.strip()

    try:
        # 만약 AI가 ```json ... ``` 형태로 주면 정규표현식이나 strip으로 제거 필요
        if "```" in raw_res:
            raw_res = raw_res.split("```")[1].replace("json", "").strip()

        res_json = json.loads(raw_res)
        script_text = res_json.get("script", "")
        highlight_word = res_json.get("highlight_word", "")
        highlight_time = res_json.get("highlight_time", 2.0)
    except:
        # 파싱 실패 시 대비 (예외 처리)
        script_text = raw_res
        highlight_word = None
        highlight_time = 2.0

    state["cur_script"] = script_text
    state['cur_highlight_word'] = highlight_word
    state['cur_highlight_time'] = highlight_time

    idx = state["slide_index"]
    state["slides"][idx]["script"] = script_text
    state["slides"][idx]["highlight_word"] = highlight_word
    state["slides"][idx]["highlight_time"] = highlight_time

    return state

* 테스트

In [ ]:
# 테스트
state = node_generate_script_ctx(state)
print(state["cur_script"][:200], "...")

In [ ]:
# 스크립트 수정 노드
def node_retouch_script(state: State) -> State:
    """
    사용자 피드백을 반영해 현재 슬라이드 스크립트를 수정하는 노드
    """

    slide_index = state["slide_index"]
    cur_slide = state["slides"][slide_index]

    script = state.get("cur_script", "")
    feedback = state.get("feedback", "")

    # 추가: state 바구니에 있는 하이라이트 임시 정보 가져오기
    h_word = state.get("cur_highlight_word", "")
    h_time = state.get("cur_highlight_time", 0.0)

    media_dir = os.path.join(state["work_dir"], "media")
    os.makedirs(media_dir, exist_ok=True)

    # 피드백이 없으면 수정 없이 그대로 진행
    if not feedback:
        cur_slide["script"] = script
        # 추가: 피드백이 없어도 하이라이트 정보는 슬라이드에 저장
        cur_slide["highlight_word"] = h_word
        cur_slide["highlight_time"] = h_time
        return state
        # return {**state, "cur_script": script}

    prompt = state["prompt"]

    sys_msg = f"""
    너는 교육용 강의 스크립트를 피드백에 맞게 수정하는 AI 편집자야.

    아래 규칙을 반드시 지켜.

    1. 기존 스크립트의 핵심 내용은 유지해.
    2. 사용자 피드백을 자연스럽게 반영해.
    3. 없는 내용을 과장하거나 추가하지 마.
    4. 여러 슬라이드가 이어지는 강의 흐름이므로 매번 인사말로 시작하지 마.
    5. 말투는 {prompt.get("tone", "친절하고 명료한 강의 톤")}을 유지해.
    6. 결과는 수정된 스크립트만 출력해.
    """

    human_msg = f"""
    [기존 스크립트]
    {script}

    [사용자 피드백]
    {feedback}

    위 피드백을 반영해서 스크립트를 자연스럽게 수정해줘.
    """

    response = llm.invoke([SystemMessage(content=sys_msg), HumanMessage(content=human_msg)])
    retouch_script = response.content

    # slides 안에 수정본 저장
    cur_slide["script"] = retouch_script
    cur_slide["feedback"] = feedback

    # 추가: 피드백이 없어도 하이라이트 정보는 슬라이드에 저장
    cur_slide["highlight_word"] = h_word
    cur_slide["highlight_time"] = h_time

    # 파일도 수정본으로 다시 저장
    script_path = os.path.join(media_dir, f"slide_{slide_index}_script_retouched.txt")

    with open(script_path, "w", encoding="utf-8") as f:
        f.write(retouch_script)

    cur_slide["script_path"] = script_path

    return {**state, "cur_script": retouch_script, "slides": state["slides"]}

### (6) 자막 타이밍 분리

In [ ]:
# 스크립트를 문장 단위로 나누고 타이밍 계산
# =============================================
def node_gen_subtitle(state: State) -> State:
    """스크립트를 자막 단위로 분리하고 타이밍 추정"""
    script = state.get("cur_script", "")
    if not script:
        state["cur_subtitles"] = []
        return state

    # 문장 분리 (split_sents 유틸 함수 활용)
    sentences = split_sents(script)

    # 총 글자 수 기반으로 타이밍 비율 계산
    # 평균 한국어 TTS 속도: 약 5~6글자/초
    CHARS_PER_SEC = 5.5
    subtitles = []
    current_time = 0.0

    for sent in sentences:
        char_count = len(sent.replace(" ", ""))
        duration = max(1.5, char_count / CHARS_PER_SEC)  # 최소 1.5초 보장
        subtitles.append({
            "text": sent,
            "start": round(current_time, 2),
            "end": round(current_time + duration, 2),
            "highlights": []  # highlight_keywords에서 채워짐
        })
        current_time += duration

    state["cur_subtitles"] = subtitles
    print(f"[subtitle] {len(subtitles)}개 자막 생성 완료")
    return state

In [ ]:
# 테스트
state = node_gen_subtitle(state)

### (7) 중요 키워드 강조 처리

In [ ]:
# LLM으로 중요 키워드 추출 및 강조 태그 추가
# =============================================
def node_highlight_keywords(state: State) -> State:
    """각 자막 문장에서 중요 키워드를 LLM으로 추출"""
    subtitles = state.get("cur_subtitles", [])
    if not subtitles:
        return state

    # 전체 스크립트를 한번에 보내서 키워드 추출 (API 절약)
    all_text = "\n".join([f"{i+1}. {s['text']}" for i, s in enumerate(subtitles)])

    messages = f"""다음은 강의 자막 문장들입니다. 각 문장에서 가장 중요한 핵심 키워드를 1~2개만 추출하세요.
반드시 아래 JSON 형식으로만 응답하세요. 다른 말은 하지 마세요.

문장들:
{all_text}

응답 형식 (문장 번호 순서대로):
{{"keywords": [["키워드1"], ["키워드1", "키워드2"], ...]}}
"""

    try:
        response = llm.invoke(messages)
        import json, re
        # JSON 파싱
        json_str = re.search(r'\{.*\}', response.content, re.DOTALL)
        if json_str:
            result = json.loads(json_str.group())
            keywords_list = result.get("keywords", [])
            for i, sub in enumerate(subtitles):
                if i < len(keywords_list):
                    sub["highlights"] = keywords_list[i]
    except Exception as e:
        print(f"[highlight] 키워드 추출 실패: {e}")

    state["cur_subtitles"] = subtitles
    print(f"[highlight] 키워드 강조 처리 완료")
    return state

In [ ]:
# 테스트
state = node_highlight_keywords(state)

## **3. 미션④ : 모듈 고도화2**

* 음성 변환 : 강의 목소리, 톤 조절 프롬프트 기반 음성 변환
* 영상 제작 : 각 슬라이드 강의를 전체 슬라이드 강의 영상으로 합치기
* 전체 Agent 그래프 구축
* 웹 화면 연결(gradio)



### (1) 음성 변환

* 목적 : 발표 스크립트(script)를 TTS 모델을 이용해 음성(mp3) 파일로 변환하고 state에 저장
* 입력
    * 발표 스크립트
    * 목소리 프리셋. 기본값 "alloy"
* 출력: 생성된 mp3 파일 및 경로

* 노드 함수 생성

In [ ]:
def node_tts(state: State) -> State:
    client = OpenAI()
    # 정보 가져오기
    script = state.get('cur_script', '')
    prompt = state.get('prompt', {})
    voice = prompt.get('voice', 'alloy')
    work_dir = state.get('work_dir', './')
    idx = state.get('slide_index', 0)
    audio_path = os.path.join(work_dir, f"narration_{idx}.mp3")

    # 스크립트가 비어있으면 에러 방지
    if not script:
        print("경고: 변환할 텍스트(cur_script)가 비어있습니다!")
        state["cur_audio"] = ""
        return state

    # 오디오 생성
    resp = client.audio.speech.create(
        model = "gpt-4o-mini-tts",
        voice = voice,
        input = script)
    # 파일 저장
    with open(audio_path, 'wb') as f:
      f.write(resp.content)
    state["cur_audio"]= audio_path
    # # 오디오 길이 측정
    # duration = ffprobe_duration(audio_path)
    # print(f"[{idx}번 슬라이드] 오디오 길이: {duration:.2f}초")
    return state

* 노드 테스트

In [ ]:
# 테스트
state = node_tts(state)
# 오디오 play 예시 코드
from IPython.display import Audio, display
audio_path = state["cur_audio"]
display(Audio(filename=audio_path))
print("Audio file:", audio_path)

### (2) 영상 제작
* 목적 : 슬라이드 이미지와 음성을 합쳐 mp4 영상 생성
* 입력: 오디오 파일, 이미지(스냅샷)
* 출력: 생성된 mp4 파일 및 경로

* 노드 함수 생성

In [ ]:
def node_make_video(state: State) -> State:
    """현재 슬라이드의 이미지(snap)와 음성(cur_audio)을 합쳐 mp4 생성"""

    # 1. 정보 준비 ---------------
    idx = state.get("slide_index", 0)
    work_dir = state.get('work_dir', './')

    # [데이터 타겟팅] 현재 슬라이드의 재료 가져오기
    curr_slide = state["slides"][idx]

    # 슬라이드 PNG (snap)
    in_png = curr_slide.get('snap', '')

    # 오디오 경로 (이전 node_tts에서 생성한 현재 오디오)
    in_mp3 = state.get('cur_audio', '')

    # [추가] 시각 효과 데이터 가져오기
    h_word = state.get('cur_highlight_word')
    h_time = state.get('cur_highlight_time', 2.0)

    print(f"[Slide {idx+1}] 영상 합성 시작...")
    # print(f"  - 이미지: {in_png}")
    # print(f"  - 오디오: {in_mp3}")

    if not in_png or not in_mp3:
        print(f"[Slide {idx+1}] 이미지 또는 음성이 없어 영상 생성을 건너뜁니다.")
        state['cur_video'] = None
        return state

    # 2. 출력 경로 설정 ---------------
    # 파일명이 중복되지 않도록 설정 (예: slide0_lecture.mp4)
    out_mp4 = os.path.join(work_dir, f'slide{idx}_lecture.mp4')

    # 3. 렌더링 (MoviePy 기반 render_mp4 함수 호출) ---------------
    try:
        # 기존에 정의된 render_mp4 함수를 그대로 사용한다고 가정합니다.
        # 이 함수 내부에서 moviepy를 이용해 합성을 수행해야 합니다.
        render_mp4(in_png, in_mp3, out_mp4,
                   highlight_word=h_word,
                   highlight_time=h_time)

        # 4. State 업데이트 ----------------
        # 현재 루프용 변수 저장
        state['cur_video'] = out_mp4

        # 누적 리스트(videos)에 저장 (마지막 병합 노드에서 사용)
        if "video_path" not in state or not state["video_path"]:
            state["video_path"] = [None] * state["n_slides"]
        state["video_path"][idx] = out_mp4

        print(f"[Slide {idx+1}] 강조 효과({h_word}) 개별 영상 생성 완료: {out_mp4}")

    except Exception as e:
        print(f"[Slide {idx+1}] 영상 렌더링 실패: {e}")
        state['cur_video'] = None
    #하드코딩 부분
    # state['slide_index'] += 1
    return state

* 노드 테스트

In [ ]:
state = node_make_video(state)

In [ ]:
state

In [ ]:
video_path = state["cur_video"]

with open(video_path, "rb") as f:
    data = f.read()

display(Video(data=data, embed=True, mimetype="video/mp4", width=960))

### (3) 자막 영상에 합성 (ffmpeg)

In [ ]:
def make_ass_subtitle(subtitles: list, font_path: str, font_size: int = 28) -> str:
    # 폰트 이름 추출 (경로에서 파일명만 추출)
    font_name = os.path.basename(font_path).split('.')[0]

    # ASS 헤더 및 스타일 정의
    header = f"""[Script Info]
ScriptType: v4.00+
PlayResX: 1280
PlayResY: 720
ScaledBorderAndShadow: yes

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,{font_name},{font_size},&H00FFFFFF,&H000000FF,&H00000000,&H00000000,0,0,0,0,100,100,0,0,1,2,2,2,10,10,50,1
"""
    #  Alignment 2는 하단 중앙, Encoding 1은 한글

    def sec_to_ass(sec):
        h = int(sec // 3600)
        m = int((sec % 3600) // 60)
        s = sec % 60
        return f"{h}:{m:02d}:{s:05.2f}"

    events = ["\n[Events]", "Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text"]

    for sub in subtitles:
        start = sec_to_ass(sub["start"])
        end   = sec_to_ass(sub["end"])

        # 줄바꿈 제거 및 정리
        text = sub["text"].replace("\n", " ").strip()
        highlights = sub.get("highlights", [])

        # 키워드 강조 처리 (긴 단어 우선순위)
        sorted_highlights = sorted(highlights, key=len, reverse=True)
        for kw in sorted_highlights:
            if not kw: continue
            # 빨간색 강조: {\c&H0000FF&}, 다시 흰색으로: {\c&HFFFFFF&}
            text = text.replace(kw, f"{{\\c&H0000FF&}}{kw}{{\\c&HFFFFFF&}}")

        events.append(f"Dialogue: 0,{start},{end},Default,,0,0,0,,{text}")

    return header + "\n".join(events) + "\n"

import subprocess
import os

def node_burn_subtitle(state: State) -> State:
    video_in   = state.get("cur_video", "")
    subtitles  = state.get("cur_subtitles", [])

    if not video_in or not subtitles or not os.path.exists(video_in):
        print("[burn_subtitle] 영상 또는 자막 없음, 스킵")
        return state

    idx      = state.get("slide_index", 0)
    work_dir = state.get("work_dir", "./")

    video_out = os.path.join(work_dir, f"slide{idx}_final_burned.mp4")
    ass_path  = os.path.join(work_dir, f"slide{idx}.ass")

    FONT_PATH = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"

    # 1. ASS 자막 파일 생성
    ass_content = make_ass_subtitle(subtitles, FONT_PATH, font_size=24)
    with open(ass_path, "w", encoding="utf-8") as f:
        f.write(ass_content)

    # 2. ffmpeg 자막 합성
    ass_escaped = ass_path.replace("\\", "/").replace(":", "\\:")
    cmd = [
        "ffmpeg", "-y",
        "-i", video_in,
        "-vf", f"ass='{ass_escaped}'",
        "-c:v", "libx264", "-preset", "veryfast",
        "-c:a", "copy",
        video_out
    ]

    # 3. video_path 리스트 준비
    video_path = state.get("video_path", [None] * state.get("n_slides", 1))
    if not isinstance(video_path, list):
        video_path = [None] * state.get("n_slides", 1)

    # 4. ffmpeg 실행 (딱 한 번만!)
    try:
        subprocess.check_call(cmd)
        # 성공 → 자막 합성된 경로로 업데이트
        video_path[idx] = video_out
        state["cur_video"] = video_out
        print(f"[burn_subtitle] ✅ 자막 합성 완료: {video_out}")
    except subprocess.CalledProcessError as e:
        print(f"[burn_subtitle] ❌ 자막 합성 실패: {e}")
        # 실패 → 자막 없는 원본 경로라도 저장 (최선책)
        video_path[idx] = video_in
        state["cur_video"] = video_in

    return {**state, "video_path": video_path}

In [ ]:
state = node_burn_subtitle(state)

### (3) 반복&종료 분기, 영상 합치기 노드
* 반복&종료 분기 : 마지막 슬라이드까지 반복 실행
* 각 슬라이드 영상을 하나의 영상으로 합치기

* State의 키(리스트)에 누적

* 분기 함수

In [ ]:
def node_acc_step(state: State) -> State:
    return{
        **state,
        "slide_index": state["slide_index"] + 1
    }

* 영상 합치기
    * concat_videos_ffmpeg 사용

In [ ]:
def node_concat_videos(state: State) -> State:
    """누적된 개별 슬라이드 영상들을 ffmpeg를 사용하여 하나의 영상으로 병합"""

    # 1. 정보 준비
    # 리스트에 순서대로 담긴 영상 경로들을 가져옵니다.
    video_list = state.get("video_path", [])
    work_dir = state.get("work_dir", "./")

    # 유효한 경로만 필터링 (혹시 모를 None 방지)
    valid_videos = [v for v in video_list if v and os.path.exists(v)]

    if not valid_videos:
        print("병합할 영상 파일이 존재하지 않습니다.")
        return state

    # 2. 출력 파일 경로 설정
    out_path = os.path.join(work_dir, "final_lecture_result.mp4")

    # 3. concat_videos_ffmpeg 호출
    print(f"총 {len(valid_videos)}개의 클립을 병합 중...")
    try:
        # 같은 코덱으로 생성되었으므로 reencode=False로 빠르게 병합 시도
        # 만약 영상 규격이 달라 오류가 나면 reencode=True로 설정하세요.
        concat_videos_ffmpeg(valid_videos, out_path, reencode=False)

        # 4. 결과 저장
        state["final_video"] = out_path
        print(f"최종 강의 영상이 생성되었습니다: {out_path}")

    except Exception as e:
        print(f"영상 병합 실패: {e}")

    return state

In [ ]:
state = node_concat_videos(state)

In [ ]:
# 동영상 play 예시 코드
video_path = state["final_video"]

with open(video_path, "rb") as f:
    data = f.read()

display(Video(data=data, embed=True, mimetype="video/mp4", width=960))

### (4) 맥락 기반 자동 퀴즈 생성기

* vector DB 사용. 한 단원(여러 개의 슬라이드)의 처리가 끝나면, Vector DB에 저장된 해당 단원의 핵심 개념(Key Concept) 벡터들을 조회
* LLM에게 객관식 퀴즈를 만들라고 지시할 때, 정답과 '벡터 거리는 가깝지만(맥락은 비슷하지만) 내용은 다른' 개념들을 Vector DB에서 가져와 오답(Distractor) 선지로 활용
* 단순히 무작위로 만든 선지보다 훨씬 더 정교하고 헷갈리는, 학습 효과가 높은 고품질 퀴즈를 자동으로 생성가능



* Vector DB 구축 노드함수:

모든 슬라이드의 파싱이 끝난 직후, 슬라이드들의 내용을 텍스트 청크로 묶어 ChromaDB에 임베딩하는 노드

In [ ]:
# 필요한 라이브러리 설치
!pip install -q langchain-openai langchain-community chromadb pymupdf

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

# vector DB 만드는 노드함
def node_build_vector_db(state: State) -> State:
    """파싱된 모든 슬라이드 내용을 바탕으로 ChromaDB를 생성하는 노드"""
    print("[Vector DB] 슬라이드 내용 임베딩 및 DB 구축 시작")
    slides = state.get("slides", [])
    docs = []

    for slide in slides:
        idx = slide.get("index", 0)
        title = slide.get("title", "")
        # 슬라이드의 문장들을 하나의 텍스트로 병합
        texts = slide.get("texts", [])
        flat_texts = []
        for t in texts:
            if isinstance(t, list):
                flat_texts.extend(t)
            else:
                flat_texts.append(t)

        content = f"제목: {title}\n내용: {' '.join(flat_texts)}"

        # LangChain Document 객체 생성 (메타데이터에 인덱스와 제목 저장)
        doc = Document(
            page_content=content,
            metadata={"slide_index": idx, "title": title}
        )
        docs.append(doc)

    # OpenAI 임베딩 모델을 사용하여 ChromaDB에 저장
    embeddings = OpenAIEmbeddings()
    # 주의: 동일 폴더에 여러 번 실행 시 덮어쓰기 위해 컬렉션 이름을 지정하거나 in-memory 모드 사용
    vectorstore = Chroma.from_documents(docs, embeddings, collection_name="lecture_slides")

    state["vectorstore"] = vectorstore
    print("[Vector DB] 구축 완료")
    return state

* 맥락 기반 퀴즈 생성 노드:

모든 영상 루프가 끝난 뒤(DONE), 영상 병합(concat) 직전에 실행되는 노드.

특정 개념과 '거리는 가깝지만 내용은 다른' 슬라이드를 조회하여 오답 선지로 만듦

In [ ]:
import random
import json

def node_generate_quiz(state: State) -> State:
    """전체 PPT 맥락에서 무작위로 타겟을 선정하고, Vector DB 오답 기반의 퀴즈를 n개 생성"""
    print("[Quiz] 맥락 기반 퀴즈 생성 시작")
    vectorstore = state["vectorstore"]
    slides = state["slides"]

    # State에서 생성할 문제 수를 가져옴 (기본값 3개로 설정)
    num_to_gen = state.get("num_quizzes_to_generate", 3)

    generated_quizzes = []

    print(f"[Quiz] 전체 슬라이드에서 핵심 개념을 추출하여 총 {num_to_gen}개의 퀴즈를 생성합니다.\n")

    # 1. 문제 수에 맞게 타겟 슬라이드 무작위 추출 (중복 방지 로직 포함)
    if num_to_gen <= len(slides):
        # 슬라이드가 문제 수보다 많거나 같으면 중복 없이 뽑기
        target_slides = random.sample(slides, num_to_gen)
    else:
        # 문제 수가 슬라이드 수보다 많으면 중복을 허용해서 뽑기
        target_slides = random.choices(slides, k=num_to_gen)

    # 2. 지정된 문제 수만큼 반복하여 퀴즈 생성
    for i, target_slide in enumerate(target_slides):
        print(f" - [{i+1}/{num_to_gen}] 번째 퀴즈 생성 중 (기준 슬라이드: {target_slide.get('index')}번)...")

        # 텍스트 병합 처리
        texts = target_slide.get("texts", [])
        flat_texts = []
        for t in texts:
            if isinstance(t, list): flat_texts.extend(t)
            else: flat_texts.append(t)

        target_content = f"제목: {target_slide.get('title', '')}\n내용: {' '.join(flat_texts)}"

        # [RAG 적용] 타겟 개념과 벡터 거리가 가까운 유사 문서를 K=4개 검색
        retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
        similar_docs = retriever.invoke(target_content)

        # 타겟 슬라이드(정답)와 동일한 슬라이드는 제외하고 오답 후보(Distractor) 추출
        distractor_candidates = []
        for doc in similar_docs:
            if doc.metadata.get("slide_index") != target_slide.get("index"):
                distractor_candidates.append(doc.page_content)

        distractor_text = "\n\n".join([f"[유사 오답 개념 {idx+1}]\n{text}" for idx, text in enumerate(distractor_candidates)])

        # LLM 프롬프트 구성
        sys_msg = """
당신은 훌륭한 AI 강의 평가자입니다. 주어진 학습 내용과 유사 오답 후보들을 바탕으로 객관식 퀴즈(4지 선다형)를 하나 생성해주세요.

[지침]
1. '학습 내용'을 기반으로 정답이 되는 질문과 정답 선지를 1개 만드세요.
2. '유사 오답 개념'으로 제공된 텍스트의 맥락(단어, 도메인)을 적극 활용하되, 사실과 다르게 교묘하게 비틀어 가장 헷갈리고 매력적인 오답 선지 3개를 만드세요. 무작위나 엉뚱한 오답은 절대 금지합니다.
3. 출력은 반드시 아래의 JSON 형식으로만 작성해야 하며, 시작과 끝에 마크다운 코드 블록(```json 및 ```)이나 부연 설명, 인사말을 절대 포함하지 마세요.
4. JSON 파싱 에러가 발생하지 않도록 큰따옴표("") 사용 및 쉼표(,) 배치에 주의하세요.

[출력 형식 (JSON)]
{
  "question": "[문제 내용]",
  "options": [
    "1. [선지 1]",
    "2. [선지 2]",
    "3. [선지 3]",
    "4. [선지 4]"
  ],
  "answer": "[정답 번호 숫자만, 예: 2]",
  "explanations": {
    "1": "[1번 선지에 대한 해설]",
    "2": "[2번 선지에 대한 해설]",
    "3": "[3번 선지에 대한 해설]",
    "4": "[4번 선지에 대한 해설]"
  }
}
"""
        human_msg = f"""
[학습 내용 - 정답 출제용]
{target_content}

[유사 오답 개념 - 매력적인 오답 생성용]
{distractor_text if distractor_text else '유사 오답 개념이 부족할 경우 맥락에 맞춰 창작해주세요.'}
"""

        # 퀴즈 생성 (LLM 호출)
        response = llm.invoke([SystemMessage(content=sys_msg), HumanMessage(content=human_msg)])

        # 문자열로 반환된 결과를 딕셔너리로 안전하게 변환하여 리스트에 추가
        try:
            quiz_dict = json.loads(response.content.strip())
            generated_quizzes.append(quiz_dict)
        except json.JSONDecodeError:
            print(f"   ⚠️ [{i+1}번 문제] JSON 파싱 에러 발생. LLM 응답을 확인하세요.")
            print(response.content)

    # 3. 완성된 퀴즈 리스트를 State에 저장
    state["quiz_results"] = generated_quizzes
    print(f"\n[Quiz] 총 {len(generated_quizzes)}개의 퀴즈 생성 완료!\n" + "-"*30)

    return state

### (4) Agent 만들기 : 그래프로 엮기
* 다음 그래프를 참조로 하나의 에이전트로 엮어 봅시다.

## Agent workflow image

> Embedded base64 image was removed for a cleaner public repository. See `docs/agent-workflow.md` for the workflow summary.


In [ ]:
from langgraph.graph import StateGraph, START, END

# 그래프 생성
builder = StateGraph(State)

# 노드 추가
builder.add_node("parse_all", node_parse_all)
builder.add_node("gen_page", node_generate_page)
builder.add_node("gen_script_ctx", node_generate_script_ctx)
builder.add_node("tts", node_tts)
builder.add_node("make_video", node_make_video)
builder.add_node("acc_step", node_acc_step)
builder.add_node("concat", node_concat_videos)
builder.add_node("tool_search", node_tool_search)
builder.add_node("retouch_script", node_retouch_script)
builder.add_node("build_vdb", node_build_vector_db)
builder.add_node("gen_quiz", node_generate_quiz)
builder.add_node("gen_subtitle", node_gen_subtitle)
builder.add_node("highlight_keywords", node_highlight_keywords)
builder.add_node("burn_subtitle", node_burn_subtitle)

# 기본 흐름 연결
builder.add_edge(START, "parse_all")
builder.add_edge("parse_all", "build_vdb")
builder.add_edge("build_vdb", "tool_search")
builder.add_edge("tool_search", "gen_page")
builder.add_edge("gen_page", "gen_script_ctx")
builder.add_edge("gen_script_ctx", "retouch_script")
builder.add_edge("retouch_script", "gen_subtitle")
builder.add_edge("gen_subtitle", "highlight_keywords")
builder.add_edge("highlight_keywords", "tts")
builder.add_edge("tts", "make_video")
builder.add_edge("make_video", "burn_subtitle")
builder.add_edge("burn_subtitle", "acc_step")

# 분기 함수
def route_after_acc(state: State):
    if state["slide_index"] < state["n_slides"]:
        return "CONTINUE"
    else:
        return "DONE"

# 조건부 연결
builder.add_conditional_edges("acc_step", route_after_acc, {"CONTINUE" : "tool_search", "DONE" : "gen_quiz"})
builder.add_edge("gen_quiz", "concat")
# 마지막 연결
builder.add_edge("concat", END)

# 컴파일
app = builder.compile()

In [ ]:
app

## **4. 시스템 실행**

미션3,4에서 수행한 결과를 통합 테스트 해 봅시다.

### (1) 준비 작업
* 파일 업로드
* 사용자 프롬프트 준비

In [ ]:
# 파일 업로드
uploaded = files.upload()
pptx_path = list(uploaded.keys())[0]

In [ ]:
# 사용자 프롬프트
USER_PROMPT = {
    "voice": "nova",
    "tone": "친절하고 명료한 강의 톤",
    "style": "예시와 핵심 요점 중심"
}
# 출력 dir 만들기
WORK_DIR = "./step2_output"
MEDIA_DIR = "./step2_output/media"
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(MEDIA_DIR, exist_ok=True)

### (2) Agent 실행
* State 초기화
* app 실행
* 동영상 play

In [ ]:
# # 코랩이나 주피터 노트북이라면 앞에 !를 붙여주세요
# !pip install --upgrade chromadb opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp

In [ ]:
feedback = input("원하는 feedback이 있으시면 말씀해주세요 (없는 경우 입력 0): ").strip()

if feedback == "0":
    feedback = ""

# 1. State 초기화
init_state: State = {
    "pptx_path": pptx_path,
    "work_dir": WORK_DIR,
    "prompt": USER_PROMPT,
    "slide_index": 0,
    "feedback": feedback,
    "num_quizzes_to_generate": 3
}

# 2. Agent 실행
result = app.invoke(init_state)

In [ ]:
# 3. 결과 확인
print("총 슬라이드 수:", result["n_slides"])
print("최종 영상 경로:", result["final_video"])
print("최종 영상 존재 여부:", os.path.exists(result["final_video"]))

In [ ]:
# 슬라이드별 생성 결과 확인
for slide in result["slides"]:
    print(f"[slide {slide['index']}]")
    print("title:", slide.get("title"))
    print("video:", slide.get("video"))

In [ ]:
# 4. 동영상 재생
display(Video(result["final_video"], embed=True, mimetype="video/mp4", width=960))

In [ ]:
import time

print("\n" + "="*50)
print("[맥락 기반 자동 퀴즈 생성 결과]")
print("="*50)

# Vector DB 구축 여부 확인
if "vectorstore" in result and result["vectorstore"] is not None:
    print("Vector DB (Chroma) 정상 구축 완료\n")
else:
    print("Vector DB 구축 실패 (또는 State에 저장되지 않음)\n")

# 퀴즈 결과 불러오기
quiz_results = result.get("quiz_results", [])

if not quiz_results:
    print("생성된 퀴즈가 없습니다. 퀴즈 생성 노드(gen_quiz)가 제대로 실행되었는지 확인해주세요.")
else:
    total_quizzes = len(quiz_results)
    print(f"총 {total_quizzes}개의 퀴즈가 준비되어 있습니다.\n")

    # 1. 풀 문제 수 설정
    while True:
        try:
            num_to_solve = int(input(f"몇 문제를 푸시겠습니까? (1 ~ {total_quizzes}): "))
            if 1 <= num_to_solve <= total_quizzes:
                break
            else:
                print(f"1에서 {total_quizzes} 사이의 숫자를 입력해주세요.")
        except ValueError:
            print("올바른 숫자를 입력해주세요.")

    print("\n" + "-"*50)
    print(f"퀴즈 풀이를 시작합니다! (총 {num_to_solve}문제)")
    print("-"*50)

    score = 0
    selected_quizzes = quiz_results[:num_to_solve] # 앞에서부터 선택한 개수만큼 가져옴

    # 2. 문제 풀이 루프
    for idx, quiz in enumerate(selected_quizzes):
        print(f"\n[문제 {idx+1} / {num_to_solve}] 💡")

        # LLM이 딕셔너리로 반환했다고 가정 (구조화된 출력)
        if isinstance(quiz, dict):
            question = quiz.get("question", "질문이 없습니다.")
            options = quiz.get("options", [])
            correct_answer = str(quiz.get("answer", "")).strip()
            explanations = quiz.get("explanations", {})

            # 문제와 선지 출력
            print(question)
            for opt in options:
                print(opt)

            # 사용자 답안 입력
            user_answer = input("\n정답을 입력하세요 (숫자): ").strip()

            print("\n채점 중", end="")
            for _ in range(3):
                time.sleep(0.3)
                print(".", end="")
            print("\n")

            # 채점
            if user_answer == correct_answer:
                print("정답입니다!")
                score += 1
            else:
                print(f"오답입니다. (정답: {correct_answer}번)")

            # 각 선지별 해설 출력
            print("\n[각 선지별 해설]")
            if isinstance(explanations, dict):
                for opt_num, exp in explanations.items():
                    print(f" - {opt_num}번: {exp}")
            else:
                print(explanations) # 딕셔너리가 아닌 통 글일 경우

        # LLM이 단순 텍스트로만 반환한 경우의 예외 처리
        else:
            print(quiz)
            user_answer = input("\n정답을 입력하세요: ")
            print("\n(단순 텍스트 모드에서는 자동 채점 및 선지별 해설 기능이 제한됩니다.)")

        print("\n" + "-"*50)

    # 최종 결과 출력
    print(f"\n최종 결과: 총 {num_to_solve}문제 중 {score}문제를 맞추셨습니다!")
    print(f"정답률: {(score/num_to_solve)*100:.1f}%\n")
    print("="*50)

## **5. Web APP 개발(Gradio)**
* 화면 개발
    * 입력 : ppt파일, 프롬프트(강의 작성톤), voice 선택(alloy, ...)
    * 출력 : 동영상 play, 동영상 다운로드 버튼
* 기본 화면을 제공합니다. 필요한 기능을 직접 추가하세요.
    * 그래프 컴파일 이름 : `app`
    * state에서 출력 비디오 이름 : `final_video`

In [ ]:
#!pip install nest_asyncio

In [ ]:
# import nest_asyncio
# 코랩/주피터노트북과 gradio의 비동기 이벤트 루프 충돌 방지
# nest_asyncio.apply()

In [ ]:
import os, time, shutil
import gradio as gr
#import nest_asyncio

# 코랩/주피터노트북과 gradio의 비동기 이벤트 루프 충돌 방지
#nest_asyncio.apply()

VOICES = ["alloy", "aria", "verse", "shimmer", "coral", "sage", "nova", "amber"]

# 1. 최초 실행 (대본 + 영상 + 퀴즈 생성)
def run_pipeline_ui(pptx_file, tone, voice):
    if pptx_file is None:
        return "PPTX 파일을 업로드해주세요.", None, None, None, []

    # 작업 디렉터리 & 업로드 저장
    work_dir = os.path.join("./webio", f"run-{int(time.time())}")
    os.makedirs(work_dir, exist_ok=True)
    pptx_path = os.path.join(work_dir, "input.pptx")
    shutil.copy(pptx_file.name, pptx_path)

    # 초기 state
    state = {
        "pptx_path": pptx_path,
        "work_dir": work_dir,
        "prompt": {
            "voice": voice or "alloy",
            "tone":  tone or "친절하고 명료한 강의 톤",
            "style": "예시와 핵심 요점 중심",
        },
        "feedback": "",
        "num_quizzes_to_generate": 3
    }

    # 그래프 실행
    final_state = app.invoke(state)

    # 🌟 대본 및 하이라이트 취합
    slides = final_state.get("slides", [])
    full_script = ""
    for s in slides:
        idx = s.get("index", 0)
        script_text = s.get("script", "(대본 없음)")
        h_word = s.get("highlight_word", "없음")
        h_time = s.get("highlight_time", 0.0)

        full_script += f"--- [슬라이드 {idx} 대본] ---\n"
        full_script += f"✨ 강조 단어: {h_word} ({h_time}초)\n"
        full_script += f"{script_text}\n\n"

    video_path = final_state.get("final_video")

    # 🌟 퀴즈 결과물 가져오기
    quiz_results = final_state.get("quiz_results", [])

    # 리턴값: [대본 텍스트, 비디오 플레이어, 다운로드 버튼, 작업 폴더 경로, 퀴즈 리스트]
    return full_script, video_path, video_path, work_dir, quiz_results


# 2. 피드백 반영 재실행
def refine_pipeline_ui(feedback, work_dir, tone, voice):
    if not work_dir:
        return "먼저 1️⃣번 버튼을 눌러 최초 생성을 진행해주세요.", None, None, []
    if not feedback.strip():
        return "피드백을 입력해주세요.", None, None, []

    pptx_path = os.path.join(work_dir, "input.pptx")

    state = {
        "pptx_path": pptx_path,
        "work_dir": work_dir,
        "prompt": {
            "voice": voice,
            "tone": tone,
            "style": "예시와 핵심 요점 중심",
        },
        "feedback": feedback,
        "num_quizzes_to_generate": 3
    }

    final_state = app.invoke(state)

    slides = final_state.get("slides", [])
    full_script = ""
    for s in slides:
        idx = s.get("index", 0)
        script_text = s.get("script", "(대본 없음)")
        h_word = s.get("highlight_word", "없음")
        h_time = s.get("highlight_time", 0.0)

        full_script += f"--- [슬라이드 {idx} 대본 (수정됨)] ---\n"
        full_script += f"✨ 강조 단어: {h_word} ({h_time}초)\n"
        full_script += f"{script_text}\n\n"

    video_path = final_state.get("final_video")
    quiz_results = final_state.get("quiz_results", [])

    return full_script, video_path, video_path, quiz_results


# 퀴즈 구동을 위한 논리 함수들
def start_quiz(quizzes):
    """'퀴즈 풀기' 또는 '다시 풀기' 버튼을 눌렀을 때 1번 문제로 초기화"""
    if not quizzes:
        return "생성된 퀴즈가 없습니다. 먼저 에이전트를 실행해주세요.", gr.update(visible=False), 0, "", gr.update(visible=False), gr.update(visible=False), gr.update(value=None)

    quiz = quizzes[0]
    q_text = quiz.get("question", "문제 내용이 없습니다.")
    opts = "\n".join(quiz.get("options", []))
    question_md = f"### 💡 문제 1 / {len(quizzes)}\n**{q_text}**\n\n{opts}"

    # 반환: [문제 텍스트, 퀴즈 그룹 UI 보이기, 현재 인덱스 0으로 리셋, 해설 창 비우기, 다음 문제 버튼 숨기기, 다시풀기 버튼 숨기기, 라디오버튼 초기화]
    return question_md, gr.update(visible=True), 0, "", gr.update(visible=False), gr.update(visible=False), gr.update(value=None)

def submit_answer(quizzes, idx, user_answer):
    """'답안 확인' 버튼을 눌렀을 때 채점 및 해설 출력"""
    if user_answer is None:
        return "⚠️ 정답 번호를 먼저 선택해주세요.", gr.update(), gr.update()

    quiz = quizzes[idx]
    ans = str(quiz.get("answer", "")).strip()
    is_correct = ans in str(user_answer)

    # 채점 결과
    result_md = "### 🎉 정답입니다!\n" if is_correct else f"### ❌ 오답입니다. (정답: {ans}번)\n"

    # 해설 출력
    exp = quiz.get("explanations", {})
    result_md += "#### 📚 [각 선지별 해설]\n"
    if isinstance(exp, dict):
        for k, v in exp.items():
            result_md += f"- **{k}번**: {v}\n"
    else:
        result_md += str(exp)

    # 마지막 문제인지 확인하여 버튼 노출 변경
    is_last = (idx == len(quizzes) - 1)

    return result_md, gr.update(visible=not is_last), gr.update(visible=is_last)

def next_question(quizzes, idx):
    """'다음 문제' 버튼을 눌렀을 때 다음 퀴즈 세팅"""
    new_idx = idx + 1
    quiz = quizzes[new_idx]
    q_text = quiz.get("question", "문제 내용이 없습니다.")
    opts = "\n".join(quiz.get("options", []))
    question_md = f"### 💡 문제 {new_idx+1} / {len(quizzes)}\n**{q_text}**\n\n{opts}"

    return question_md, new_idx, "", gr.update(visible=False), gr.update(visible=False), gr.update(value=None)


# -------------------- Gradio UI --------------------
with gr.Blocks(title="AI 강사 Agent") as demo:
    gr.Markdown("### 🎓 PPT → 강의영상 자동 제작 에이전트")

    with gr.Row():
        inp_ppt = gr.File(label="PPTX 업로드", file_types=[".pptx"])
        with gr.Column():
            inp_tone  = gr.Textbox(value="친절하고 명료한 강의 톤", label="강의 톤")
            inp_voice = gr.Dropdown(VOICES, value="alloy", label="TTS Voice")

    run_btn = gr.Button("1️⃣ 영상 및 대본 최초 생성", variant="primary")

    out_script = gr.Textbox(label="📄 생성된 대본 확인", lines=10, interactive=False)

    inp_feedback = gr.Textbox(label="💬 대본 수정 피드백 (예: 2번 슬라이드 설명을 조금 더 쉽게 해줘)", lines=2)
    refine_btn = gr.Button("2️⃣ 피드백 반영하여 다시 만들기", variant="secondary")

    out_video = gr.Video(label="🎬 최종 동영상 미리보기")
    out_download = gr.DownloadButton(label="동영상 다운로드")

    # 퀴즈 UI
    gr.Markdown("---")
    gr.Markdown("### 📝 복습 퀴즈")
    quiz_start_btn = gr.Button("🚀 퀴즈 풀기", variant="primary")

    # 처음엔 숨겨져 있다가 '퀴즈 풀기'를 누르면 나타나는 영역
    with gr.Group(visible=False) as quiz_group:
        quiz_display = gr.Markdown("문제가 여기에 표시됩니다.")

        # 답안 선택지 (1~4번)
        quiz_answer_input = gr.Radio(choices=["1", "2", "3", "4"], label="정답 선택")
        quiz_submit_btn = gr.Button("✅ 답안 입력", variant="secondary")

        # 해설 및 결과 노출
        quiz_result_display = gr.Markdown("")

        with gr.Row():
            quiz_next_btn = gr.Button("➡️ 다음 문제", visible=False)
            quiz_retry_btn = gr.Button("🔄 한 번 더 다시 풀기", visible=False, variant="primary")

    # 상태 유지 변수들
    state_workdir = gr.State()
    state_quizzes = gr.State([]) # 생성된 퀴즈들을 리스트로 저장
    state_quiz_idx = gr.State(0) # 현재 풀고 있는 퀴즈 번호


    # 이벤트 연결

    # 1. 파이프라인 실행 시 퀴즈 결과(state_quizzes) 받아오기
    run_btn.click(
        fn=run_pipeline_ui,
        inputs=[inp_ppt, inp_tone, inp_voice],
        outputs=[out_script, out_video, out_download, state_workdir, state_quizzes]
    )

    refine_btn.click(
        fn=refine_pipeline_ui,
        inputs=[inp_feedback, state_workdir, inp_tone, inp_voice],
        outputs=[out_script, out_video, out_download, state_quizzes]
    )

    # 2. 퀴즈 조작 이벤트
    # '퀴즈 풀기' 및 '다시 풀기' 버튼
    for btn in [quiz_start_btn, quiz_retry_btn]:
        btn.click(
            fn=start_quiz,
            inputs=[state_quizzes],
            outputs=[quiz_display, quiz_group, state_quiz_idx, quiz_result_display, quiz_next_btn, quiz_retry_btn, quiz_answer_input]
        )

    # '답안 입력' 버튼
    quiz_submit_btn.click(
        fn=submit_answer,
        inputs=[state_quizzes, state_quiz_idx, quiz_answer_input],
        outputs=[quiz_result_display, quiz_next_btn, quiz_retry_btn]
    )

    # '다음 문제' 버튼
    quiz_next_btn.click(
        fn=next_question,
        inputs=[state_quizzes, state_quiz_idx],
        outputs=[quiz_display, state_quiz_idx, quiz_result_display, quiz_next_btn, quiz_retry_btn, quiz_answer_input]
    )

# Colab 환경 충돌 방지용 옵션
demo.launch(share=True, debug=True)

In [ ]:
# ### 추가 전 Gradio UI 원본 ###

# import os, time, shutil
# import gradio as gr

# VOICES = ["alloy", "aria", "verse", "shimmer", "coral", "sage", "nova", "amber"]


# # 최초 실행 (대본 + 영상 생성)
# def run_pipeline_ui(pptx_file, tone, voice):

#     # 작업 디렉터리 & 업로드 저장
#     work_dir = os.path.join("./webio", f"run-{int(time.time())}")
#     os.makedirs(work_dir, exist_ok=True)
#     pptx_path = os.path.join(work_dir, "input.pptx")
#     shutil.copy(pptx_file.name, pptx_path)

#     # 초기 state (State 스키마에 맞춤)
#     state = {
#         "pptx_path": pptx_path,
#         "work_dir": work_dir,
#         "prompt": {
#             "voice": voice or "alloy",
#             "tone":  tone or "친절하고 명료한 강의 톤",
#             "style": "예시와 핵심 요점 중심",
#         },
#     }

#     # 그래프 실행
#     final_state = app.invoke(state)
#     video_path = final_state.get("final_video")

#     return video_path, video_path

# # -------------------- Gradio UI --------------------
# with gr.Blocks(title="AI 강사 Agent") as demo:
#     gr.Markdown("### PPT --> 강의영상 자동 제작")

#     with gr.Row():
#         inp_ppt = gr.File(label="PPTX 업로드", file_types=[".pptx"])
#         with gr.Column():
#             inp_tone  = gr.Textbox(value="친절하고 명료한 강의 톤", label="강의 작성 톤 (프롬프트)")
#             inp_voice = gr.Dropdown(VOICES, value="alloy", label="TTS Voice")

#     run_btn = gr.Button("실행", variant="primary")

#     out_video    = gr.Video(label="최종 동영상 미리보기", interactive=False)
#     out_download = gr.DownloadButton(label="동영상 다운로드")

#     run_btn.click(run_pipeline_ui, [inp_ppt, inp_tone, inp_voice], [out_video, out_download])

# demo.launch()